# HAProxy — Load Balancing

HAProxy provides high-performance TCP and HTTP load balancing with health checks, ACLs, and SSL termination.

## Generate an HTTP load balancer config

In [ ]:
def haproxy_http(servers, frontend_port=80, algorithm="roundrobin", health_path="/healthz"):
    server_lines = "\n    ".join(
        f"server {name} {addr} check inter 5s rise 2 fall 3"
        for name, addr in servers.items()
    )
    return f"""global
    log stdout format raw local0 info
    maxconn 50000

defaults
    log     global
    mode    http
    option  httplog
    option  forwardfor
    timeout connect 5s
    timeout client  30s
    timeout server  30s

frontend http_front
    bind *:{frontend_port}
    default_backend app_back

backend app_back
    balance {algorithm}
    option httpchk GET {health_path}
    http-check expect status 200
    {server_lines}

frontend stats
    bind *:8404
    stats enable
    stats uri /stats
    stats refresh 10s
"""

cfg = haproxy_http(
    servers={"web1": "10.0.0.1:8080", "web2": "10.0.0.2:8080", "web3": "10.0.0.3:8080"},
    algorithm="leastconn",
)
print(cfg)

## Validate with Docker

In [ ]:
import pathlib, subprocess

cfg_path = pathlib.Path("/tmp/haproxy-generated.cfg")
cfg_path.write_text(haproxy_http(
    {"web1": "127.0.0.1:8080", "web2": "127.0.0.2:8080"},
))

result = subprocess.run(
    ["docker", "run", "--rm",
     "-v", f"{cfg_path}:/usr/local/etc/haproxy/haproxy.cfg:ro",
     "haproxy:lts-alpine", "haproxy", "-c", "-f", "/usr/local/etc/haproxy/haproxy.cfg"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

## ACL routing builder

In [ ]:
def haproxy_acl_rules(rules):
    """
    rules: list of (acl_name, condition, backend_name)
    """
    acl_lines = "\n    ".join(f"acl {name} {cond}" for name, cond, _ in rules)
    use_lines = "\n    ".join(f"use_backend {be} if {name}" for name, _, be in rules)
    return f"""    {acl_lines}
    {use_lines}
    default_backend default_back"""

rules = [
    ("is_api",    "path_beg /api/",           "api_back"),
    ("is_static", "path_beg /static/ /assets/","static_back"),
    ("is_admin",  "hdr(host) -i admin.example.com", "admin_back"),
]
print(haproxy_acl_rules(rules))